# 2D Gaussian single-cell-style simulation

Samples \(x \in \mathbb{R}^2\) with context \(c\) and perturbation \(p\). Means follow a pluggable coupling (additive, rotated, or Hadamard interaction); see `data/simulate.py`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

_repo = Path.cwd()
while _repo != _repo.parent:
    if (_repo / "data" / "simulate.py").is_file():
        break
    _repo = _repo.parent
else:
    raise FileNotFoundError("Could not locate data/simulate.py from this notebook path.")

sys.path.insert(0, str(_repo.resolve()))

from data.simulate import (
    GaussianSCDataset,
    mean_grid,
    sample_gaussian_sc,
    make_additive_coupling,
    make_hadamard_interaction_coupling,
    make_rotated_additive_coupling,
)

rng = np.random.default_rng(0)

In [ ]:
import matplotlib.lines as mlines


def scatter_by_cp(ds: GaussianSCDataset, title: str) -> None:
    fig, ax = plt.subplots(figsize=(7, 6))
    n_c = int(ds.c.max()) + 1
    n_p = int(ds.p.max()) + 1
    cmap = plt.get_cmap("tab10")
    markers = ["o", "s", "^", "D", "v", "P", "X", "*"]
    for c in range(n_c):
        for p in range(n_p):
            m = (ds.c == c) & (ds.p == p)
            ax.scatter(
                ds.x[m, 0],
                ds.x[m, 1],
                s=16,
                alpha=0.55,
                color=cmap(c % 10),
                marker=markers[p % len(markers)],
                edgecolors="none",
            )
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("gene / feature 1")
    ax.set_ylabel("gene / feature 2")
    ax.set_title(title)
    c_handles = [
        mlines.Line2D([], [], color=cmap(i % 10), marker="o", linestyle="None", markersize=8, label=f"c={i}")
        for i in range(n_c)
    ]
    p_handles = [
        mlines.Line2D(
            [],
            [],
            color="0.35",
            marker=markers[p % len(markers)],
            linestyle="None",
            markersize=8,
            label=f"p={p}",
        )
        for p in range(n_p)
    ]
    leg_c = ax.legend(handles=c_handles, title="context c", loc="upper left")
    ax.add_artist(leg_c)
    ax.legend(handles=p_handles, title="perturbation p", loc="lower left")
    plt.tight_layout()
    plt.show()


def plot_means(coupling, title: str) -> None:
    cg, pg, mu = mean_grid(coupling)
    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = plt.get_cmap("tab10")
    for c in range(coupling.n_contexts):
        sel = cg == c
        ax.plot(mu[sel, 0], mu[sel, 1], "-o", color=cmap(c % 10), lw=1.5, markersize=8, label=f"c={c}")
    for i, (mx, my) in enumerate(mu):
        ax.annotate(f"p={pg[i]}", (mx, my), textcoords="offset points", xytext=(4, 4), fontsize=8)
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("gene / feature 1")
    ax.set_ylabel("gene / feature 2")
    ax.set_title(title)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

## Additive means \(\mu_{c,p} = \mu_c + \delta_p\)

In [ ]:
coupling_add = make_additive_coupling(
    rng,
    n_contexts=3,
    n_perturbations=4,
    dim=2,
    context_scale=1.2,
    pert_scale=0.9,
)
plot_means(coupling_add, "Means: additive (polyline per context over p)")

ds_add = sample_gaussian_sc(coupling_add, samples_per_combo=200, rng=rng, sigma=0.18)
scatter_by_cp(ds_add, "Samples: additive coupling, σ=0.18")

## Rotated perturbation \(\mu_{c,p} = \mu_c + R_c \delta_p\)

Same \(\delta_p\) vectors as in the additive construction are **rotated** per context, so perturbation directions change with \(c\).

In [ ]:
rng_rot = np.random.default_rng(1)
coupling_rot = make_rotated_additive_coupling(
    rng_rot,
    # rng,
    n_contexts=3,
    n_perturbations=4,
    dim=2,
    context_scale=1.2,
    pert_scale=0.9,
)
plot_means(coupling_rot, "Means: rotated additive")

ds_rot = sample_gaussian_sc(coupling_rot, samples_per_combo=200, rng=rng_rot, sigma=0.18)
scatter_by_cp(ds_rot, "Samples: rotated additive")

## Hadamard interaction \(\mu_{c,p} = \mu_c + \delta_p + u_c \odot v_p\)

In [ ]:
rng_h = np.random.default_rng(2)
coupling_h = make_hadamard_interaction_coupling(
    rng_h,
    # rng,
    n_contexts=3,
    n_perturbations=4,
    dim=2,
    context_scale=1.0,
    pert_scale=0.7,
    interaction_scale=0.5,
)
plot_means(coupling_h, "Means: additive + Hadamard interaction")

ds_h = sample_gaussian_sc(coupling_h, samples_per_combo=200, rng=rng_h, sigma=0.18)
scatter_by_cp(ds_h, "Samples: Hadamard interaction")